# Sensitivity Analysis: Successful-Detection Subset

**Purpose.** The threats-to-validity section reports that ~20.6% of RoBERTa-based detection attempts failed (text too short), concentrated in the open-source models. This raises the question of whether the Prelim-RQ / RQ1 conclusions ("AI is largely indistinguishable, especially under linked-issue context; code-dense contexts are more detectable; GPT-5-mini is undetectable") are driven by genuine indistinguishability or by detector failure on short outputs.

**Approach.** Re-run the Prelim-RQ and RQ1 analyses on three subsets, side-by-side, and check whether the qualitative story changes:

1. **Original (no filter)** — exact reproduction of the paper's numbers; sanity check.
2. **Per-detector successful subset** — for each tool, keep only rows where that tool succeeded (`{tool}_success == True`).
3. **Common successful subset** — keep only rows where *all three* tools succeeded.

Both subsets enforce **paired filtering**: a PR is retained only if both its AI and Human rows survived the filter for the relevant tool/subset. This keeps the human baseline aligned with the AI rows it is being compared against, otherwise paired metrics get noisy.

**Source.** `phase3/datasets/temperature_1/run_3/detection_results.csv` (same file the paper's RQ1 notebook reads). Outputs land in `phase3/analysis/sensitivity_successful_only/outputs/` and do **not** overwrite anything in `phase3/analysis/outputs/`.

**Failure profile in run_3 (verified):** RoBERTa = 1,201 failures each; ZeroGPT = 0 failures. So the per-detector view differs from the no-filter view only for the two RoBERTa detectors, and the common subset = (RoBERTa-successful) ∩ (RoBERTa-QA-successful) ∩ all rows.

In [1]:
import pandas as pd
import numpy as np
import itertools
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy.stats import chi2_contingency

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Output dir:', OUTPUT_DIR.resolve())

Output dir: /Users/ayo/Documents/ayo-workspace/iwadi/llm-in-prs/phase3/analysis/sensitivity_successful_only/outputs


## Helpers (mirroring the original notebook)

In [2]:
def cliffs_delta(lst1, lst2):
    """Cliff's Delta effect size (O(m*n) reference impl, matches original notebook)."""
    m, n = len(lst1), len(lst2)
    if m == 0 or n == 0:
        return 0.0
    dom = 0
    for x in lst1:
        for y in lst2:
            if x > y:
                dom += 1
            elif x < y:
                dom -= 1
    return dom / (m * n)

def interpret_cliffs_delta(d):
    d = abs(d)
    if d < 0.147:
        return 'Negligible'
    if d < 0.33:
        return 'Small'
    if d < 0.474:
        return 'Medium'
    return 'Large'

def cramers_v_from_table(table):
    chi2, _, _, _ = chi2_contingency(table)
    table = np.asarray(table)
    n = table.sum()
    if n == 0:
        return 0.0
    r, c = table.shape
    k = min(r - 1, c - 1)
    if k <= 0:
        return 0.0
    return np.sqrt(chi2 / (n * k))

def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'Specificity': specificity,
    }

## Load data

In [3]:
CSV_PATH = '../../datasets/temperature_1/run_3/detection_results.csv'
TOOLS = ['roberta', 'roberta-qa', 'zerogpt']
VARIATION_ORDER = ['P-1', 'P-2', 'P-3', 'P-4', 'P-5', 'P-6']
VARIATION_MAP = {
    'P-1': 'Repo Path',
    'P-2': 'PR Title',
    'P-3': 'File Diff',
    'P-4': 'File Contents',
    'P-5': 'Linked Issue',
    'P-6': 'Without File Contents',
}

df = pd.read_csv(CSV_PATH)

for tool in TOOLS:
    col = f'{tool}_is_ai'
    if col in df.columns:
        df[col] = df[col].replace({'TRUE': True, 'FALSE': False, 'True': True, 'False': False})
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    sc = f'{tool}_success'
    if sc in df.columns:
        df[sc] = df[sc].astype(bool)

df['normalized_pr_id'] = df['pr_id'].astype(str)
df['Context_Name'] = df['variation'].map(VARIATION_MAP)

print('Shape:', df.shape)
print('entry_type:', df['entry_type'].value_counts().to_dict())
for tool in TOOLS:
    print(f'{tool}_success ->', df[f'{tool}_success'].value_counts().to_dict())

Shape: (5836, 22)
entry_type: {'ai': 5528, 'human': 308}
roberta_success -> {True: 4635, False: 1201}
roberta-qa_success -> {True: 4635, False: 1201}
zerogpt_success -> {True: 5836}


## Build the three views

Each view returns `(ai_df, human_df, label)` where `human_df['model'] = 'Human'` and `human_df['true_label'] = 0` and `ai_df['true_label'] = 1`. PRs are paired (intersection of AI and human PR IDs that survive the filter).

In [4]:
def build_view(df, success_cols=None, label=''):
    """Filter df to rows where the listed *_success columns are all True (if any), then pair AI/human PR IDs."""
    filtered = df.copy()
    if success_cols:
        for sc in success_cols:
            filtered = filtered[filtered[sc] == True]
    ai = filtered[filtered['entry_type'] == 'ai'].copy()
    hum = filtered[filtered['entry_type'] == 'human'].copy()
    ai_ids = set(ai['normalized_pr_id'].unique())
    hum_ids = set(hum['normalized_pr_id'].unique())
    valid = ai_ids & hum_ids
    ai = ai[ai['normalized_pr_id'].isin(valid)].copy()
    hum = hum[hum['normalized_pr_id'].isin(valid)].copy()
    ai['true_label'] = 1
    hum['true_label'] = 0
    hum['model'] = 'Human'
    return ai, hum, label, len(valid)

views = {
    'original': build_view(df, success_cols=None, label='Original (no filter)'),
    'roberta_success': build_view(df, success_cols=['roberta_success'], label='RoBERTa-success only'),
    'roberta-qa_success': build_view(df, success_cols=['roberta-qa_success'], label='RoBERTa-QA-success only'),
    'zerogpt_success': build_view(df, success_cols=['zerogpt_success'], label='ZeroGPT-success only'),
    'common_success': build_view(df, success_cols=['roberta_success', 'roberta-qa_success', 'zerogpt_success'], label='All three tools succeeded'),
}

view_summary_rows = []
for key, (ai, hum, label, n_pairs) in views.items():
    view_summary_rows.append({
        'view': key,
        'label': label,
        'paired_PRs': n_pairs,
        'ai_rows': len(ai),
        'human_rows': len(hum),
    })
view_summary = pd.DataFrame(view_summary_rows)
view_summary.to_csv(OUTPUT_DIR / 'view_summary.csv', index=False)
display(view_summary)

,view,label,paired_PRs,ai_rows,human_rows
0,original,Original (no filter),308,5528,308
1,roberta_success,RoBERTa-success only,263,3808,263
2,roberta-qa_success,RoBERTa-QA-success only,263,3808,263
3,zerogpt_success,ZeroGPT-success only,308,5528,308
4,common_success,All three tools succeeded,263,3808,263


## Prelim-RQ — baseline detection on the File Diff (P-3) context

For each (view, tool, model) we report Accuracy / Precision / Recall / F1 / Specificity, plus Chi-square + Cramer's V across models for that tool.

**How to read the views per detector:** the headline number for tool T in the rerun is the **T_success view** (apples-to-apples for that tool). The **common-success view** is the most conservative across-tool comparison.

In [5]:
def prelim_rq_for_view(ai, hum, view_label):
    rows = []
    ai_p3 = ai[ai['variation'] == 'P-3']
    models = sorted(m for m in ai_p3['model'].unique() if m != 'Human')
    for model in models:
        model_ai = ai_p3[ai_p3['model'] == model]
        # Pair humans down to PRs that exist in this model's P-3 subset
        paired_humans = hum[hum['normalized_pr_id'].isin(model_ai['normalized_pr_id'].unique())]
        combined = pd.concat([paired_humans, model_ai], ignore_index=True)
        for tool in TOOLS:
            y_true = combined['true_label']
            y_pred = combined[f'{tool}_is_ai']
            m = calculate_metrics(y_true, y_pred)
            m.update({
                'view': view_label,
                'tool': tool,
                'model': model,
                'n_human': len(paired_humans),
                'n_ai': len(model_ai),
            })
            rows.append(m)
    return pd.DataFrame(rows)

prelim_frames = []
for key, (ai, hum, label, _) in views.items():
    prelim_frames.append(prelim_rq_for_view(ai, hum, key))
prelim_all = pd.concat(prelim_frames, ignore_index=True)
prelim_all = prelim_all[['view', 'tool', 'model', 'n_human', 'n_ai', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']]
prelim_all = prelim_all.round(4)
prelim_all.to_csv(OUTPUT_DIR / 'prelim_rq_metrics_by_view.csv', index=False)
display(prelim_all)

,view,tool,model,n_human,n_ai,Accuracy,Precision,Recall,F1-Score,Specificity
0,original,roberta,Qwen/Qwen2.5-72B-Instruct,308,308,0.6006,1.0,0.2013,0.3351,1.0
1,original,roberta-qa,Qwen/Qwen2.5-72B-Instruct,308,308,0.7094,1.0,0.4188,0.5904,1.0
2,original,zerogpt,Qwen/Qwen2.5-72B-Instruct,308,308,0.7192,1.0,0.4383,0.6095,1.0
3,original,roberta,gpt-5-mini-2025-08-07,308,308,0.5032,1.0,0.0065,0.0129,1.0
4,original,roberta-qa,gpt-5-mini-2025-08-07,308,308,0.5000,0.0,0.0000,0.0000,1.0
5,original,zerogpt,gpt-5-mini-2025-08-07,308,308,0.5503,1.0,0.1006,0.1829,1.0
6,original,roberta,meta-llama/llama-3.1-70b-instruct,308,308,0.5438,1.0,0.0877,0.1612,1.0
7,original,roberta-qa,meta-llama/llama-3.1-70b-instruct,308,308,0.5406,1.0,0.0812,0.1502,1.0
8,original,zerogpt,meta-llama/llama-3.1-70b-instruct,308,308,0.5065,1.0,0.0130,0.0256,1.0
9,roberta_success,roberta,Qwen/Qwen2.5-72B-Instruct,260,260,0.6058,1.0,0.2115,0.3492,1.0


In [6]:
# Side-by-side: F1 by (tool, model) across views
f1_pivot = prelim_all.pivot_table(index=['tool', 'model'], columns='view', values='F1-Score')
f1_pivot = f1_pivot[['original', 'roberta_success', 'roberta-qa_success', 'zerogpt_success', 'common_success']]
f1_pivot.to_csv(OUTPUT_DIR / 'prelim_rq_f1_pivot.csv')
display(f1_pivot)

recall_pivot = prelim_all.pivot_table(index=['tool', 'model'], columns='view', values='Recall')
recall_pivot = recall_pivot[['original', 'roberta_success', 'roberta-qa_success', 'zerogpt_success', 'common_success']]
recall_pivot.to_csv(OUTPUT_DIR / 'prelim_rq_recall_pivot.csv')
display(recall_pivot)

view                                          original  roberta_success  roberta-qa_success  zerogpt_success  common_success
tool       model                                                                                                            
roberta    Qwen/Qwen2.5-72B-Instruct            0.3351           0.3492              0.3492           0.3351          0.3492
           gpt-5-mini-2025-08-07                0.0129           0.0151              0.0151           0.0129          0.0151
           meta-llama/llama-3.1-70b-instruct    0.1612           0.3750              0.3750           0.1612          0.3750
roberta-qa Qwen/Qwen2.5-72B-Instruct            0.5904           0.6096              0.6096           0.5904          0.6096
           gpt-5-mini-2025-08-07                0.0000           0.0000              0.0000           0.0000          0.0000
           meta-llama/llama-3.1-70b-instruct    0.1502           0.3286              0.3286           0.1502          0.3286
zerogpt    Qwen/Qwen2.5-72B-Instruct            0.6095           0.6352              0.6352           0.6095          0.6352
           gpt-5-mini-2025-08-07                0.1829           0.1924              0.1924           0.1829          0.1924
           meta-llama/llama-3.1-70b-instruct    0.0256           0.0661              0.0661           0.0256          0.0661

view                                          original  roberta_success  roberta-qa_success  zerogpt_success  common_success
tool       model                                                                                                            
roberta    Qwen/Qwen2.5-72B-Instruct            0.2013           0.2115              0.2115           0.2013          0.2115
           gpt-5-mini-2025-08-07                0.0065           0.0076              0.0076           0.0065          0.0076
           meta-llama/llama-3.1-70b-instruct    0.0877           0.2308              0.2308           0.0877          0.2308
roberta-qa Qwen/Qwen2.5-72B-Instruct            0.4188           0.4385              0.4385           0.4188          0.4385
           gpt-5-mini-2025-08-07                0.0000           0.0000              0.0000           0.0000          0.0000
           meta-llama/llama-3.1-70b-instruct    0.0812           0.1966              0.1966           0.0812          0.1966
zerogpt    Qwen/Qwen2.5-72B-Instruct            0.4383           0.4654              0.4654           0.4383          0.4654
           gpt-5-mini-2025-08-07                0.1006           0.1065              0.1065           0.1006          0.1065
           meta-llama/llama-3.1-70b-instruct    0.0130           0.0342              0.0342           0.0130          0.0342

In [7]:
# Chi-square + Cramer's V across models, per (view, tool). Mirrors paper's Table 3.
chi_rows = []
for key, (ai, hum, label, _) in views.items():
    ai_p3 = ai[ai['variation'] == 'P-3']
    models = sorted(m for m in ai_p3['model'].unique() if m != 'Human')
    for tool in TOOLS:
        contingency = []
        for model in models:
            model_ai = ai_p3[ai_p3['model'] == model]
            paired_humans = hum[hum['normalized_pr_id'].isin(model_ai['normalized_pr_id'].unique())]
            combined = pd.concat([paired_humans, model_ai], ignore_index=True)
            y_true = combined['true_label']
            y_pred = combined[f'{tool}_is_ai']
            correct = int((y_true == y_pred).sum())
            incorrect = int((y_true != y_pred).sum())
            contingency.append([correct, incorrect])
        chi2, p, dof, _ = chi2_contingency(contingency)
        v = cramers_v_from_table(contingency)
        chi_rows.append({'view': key, 'tool': tool, 'chi2': round(chi2, 4), 'dof': dof, 'p_value': p, 'cramers_v': round(v, 4)})
chi_df = pd.DataFrame(chi_rows)
chi_df.to_csv(OUTPUT_DIR / 'prelim_rq_chi_square_by_view.csv', index=False)
display(chi_df)

,view,tool,chi2,dof,p_value,cramers_v
0,original,roberta,11.9121,2,2.590155e-03,0.0803
1,original,roberta-qa,62.5202,2,2.654032e-14,0.1839
2,original,zerogpt,64.3111,2,1.083979e-14,0.1865
3,roberta_success,roberta,13.9515,2,9.342541e-04,0.1044
4,roberta_success,roberta-qa,52.7772,2,3.463922e-12,0.2031
5,roberta_success,zerogpt,48.4759,2,2.975762e-11,0.1946
6,roberta-qa_success,roberta,13.9515,2,9.342541e-04,0.1044
7,roberta-qa_success,roberta-qa,52.7772,2,3.463922e-12,0.2031
8,roberta-qa_success,zerogpt,48.4759,2,2.975762e-11,0.1946
9,zerogpt_success,roberta,11.9121,2,2.590155e-03,0.0803


## RQ1 — F1 across the six contexts and pairwise Cliff's δ

We compute F1 per (view, tool, model, context) so we can compare directly to the paper's `rq2_f1_scores.pdf` and to Table 4 (`tab:rq2_significant_effects`). Then we recompute pairwise Cliff's δ on the detector probability scores (`{tool}_ai_probability`) per (view, tool, model), and check whether the medium/large effect rows from Table 4 hold in each subset view.

In [8]:
def f1_across_contexts(ai, hum, view_label):
    rows = []
    models = sorted(m for m in ai['model'].unique() if m != 'Human')
    for var_code in VARIATION_ORDER:
        for model in models:
            ai_slice = ai[(ai['variation'] == var_code) & (ai['model'] == model)]
            if ai_slice.empty:
                continue
            paired_humans = hum[hum['normalized_pr_id'].isin(ai_slice['normalized_pr_id'].unique())]
            combined = pd.concat([paired_humans, ai_slice], ignore_index=True)
            for tool in TOOLS:
                m = calculate_metrics(combined['true_label'], combined[f'{tool}_is_ai'])
                rows.append({
                    'view': view_label,
                    'tool': tool,
                    'model': model,
                    'variation': var_code,
                    'context': VARIATION_MAP[var_code],
                    'n_human': len(paired_humans),
                    'n_ai': len(ai_slice),
                    **m,
                })
    return pd.DataFrame(rows)

f1_frames = []
for key, (ai, hum, label, _) in views.items():
    f1_frames.append(f1_across_contexts(ai, hum, key))
f1_all = pd.concat(f1_frames, ignore_index=True)
f1_all = f1_all.round(4)
f1_all.to_csv(OUTPUT_DIR / 'rq1_f1_by_view_tool_model_context.csv', index=False)
print('rows:', len(f1_all))
display(f1_all.head(20))

rows: 270


,view,tool,model,variation,context,n_human,n_ai,Accuracy,Precision,Recall,F1-Score,Specificity
0,original,roberta,Qwen/Qwen2.5-72B-Instruct,P-1,Repo Path,308,308,0.6153,1.0,0.2305,0.3747,1.0
1,original,roberta-qa,Qwen/Qwen2.5-72B-Instruct,P-1,Repo Path,308,308,0.6510,1.0,0.3019,0.4638,1.0
2,original,zerogpt,Qwen/Qwen2.5-72B-Instruct,P-1,Repo Path,308,308,0.7532,1.0,0.5065,0.6724,1.0
3,original,roberta,gpt-5-mini-2025-08-07,P-1,Repo Path,308,308,0.5000,0.0,0.0000,0.0000,1.0
4,original,roberta-qa,gpt-5-mini-2025-08-07,P-1,Repo Path,308,308,0.5000,0.0,0.0000,0.0000,1.0
5,original,zerogpt,gpt-5-mini-2025-08-07,P-1,Repo Path,308,308,0.6380,1.0,0.2760,0.4326,1.0
6,original,roberta,meta-llama/llama-3.1-70b-instruct,P-1,Repo Path,308,308,0.5942,1.0,0.1883,0.3169,1.0
7,original,roberta-qa,meta-llama/llama-3.1-70b-instruct,P-1,Repo Path,308,308,0.5747,1.0,0.1494,0.2599,1.0
8,original,zerogpt,meta-llama/llama-3.1-70b-instruct,P-1,Repo Path,308,308,0.5130,1.0,0.0260,0.0506,1.0
9,original,roberta,Qwen/Qwen2.5-72B-Instruct,P-2,PR Title,308,308,0.7305,1.0,0.4610,0.6311,1.0


In [9]:
# Pivot: F1 per (tool, model, context) across views — easiest side-by-side read
f1_context_pivot = f1_all.pivot_table(index=['tool', 'model', 'context'], columns='view', values='F1-Score')
ordered_views = ['original', 'roberta_success', 'roberta-qa_success', 'zerogpt_success', 'common_success']
f1_context_pivot = f1_context_pivot[ordered_views]
f1_context_pivot.to_csv(OUTPUT_DIR / 'rq1_f1_context_pivot.csv')
display(f1_context_pivot)

view                                                                original  roberta_success  roberta-qa_success  zerogpt_success  common_success
tool       model                             context                                                                                              
roberta    Qwen/Qwen2.5-72B-Instruct         File Contents            0.5375           0.5531              0.5531           0.5375          0.5531
                                             File Diff                0.3351           0.3492              0.3492           0.3351          0.3492
                                             Linked Issue             0.3441           0.3248              0.3248           0.3441          0.3248
                                             PR Title                 0.6311           0.7922              0.7922           0.6311          0.7922
                                             Repo Path                0.3747           0.3859              0.3859           0.3747          0.3859
                                             Without File Contents    0.3077           0.2987              0.2987           0.3077          0.2987
           gpt-5-mini-2025-08-07             File Contents            0.0066           0.0000              0.0000           0.0066          0.0000
                                             File Diff                0.0129           0.0151              0.0151           0.0129          0.0151
                                             Linked Issue             0.0065           0.0000              0.0000           0.0065          0.0000
                                             PR Title                 0.0129           0.0151              0.0151           0.0129          0.0151
                                             Repo Path                0.0000           0.0000              0.0000           0.0000          0.0000
                                             Without File Contents    0.0065           0.0076              0.0076           0.0065          0.0076
           meta-llama/llama-3.1-70b-instruct File Contents            0.5794           0.8286              0.8286           0.5794          0.8286
                                             File Diff                0.1612           0.3750              0.3750           0.1612          0.3750
                                             Linked Issue             0.1667           0.3146              0.3146           0.1667          0.3146
                                             PR Title                 0.1333           0.9756              0.9756           0.1333          0.9756
                                             Repo Path                0.3169           0.7368              0.7368           0.3169          0.7368
                                             Without File Contents    0.3660           0.4701              0.4701           0.3660          0.4701
roberta-qa Qwen/Qwen2.5-72B-Instruct         File Contents            0.8625           0.8739              0.8739           0.8625          0.8739
                                             File Diff                0.5904           0.6096              0.6096           0.5904          0.6096
                                             Linked Issue             0.2550           0.2697              0.2697           0.2550          0.2697
                                             PR Title                 0.6491           0.8228              0.8228           0.6491          0.8228
                                             Repo Path                0.4638           0.4695              0.4695           0.4638          0.4695
                                             Without File Contents    0.5121           0.5239              0.5239           0.5121          0.5239
           gpt-5-mini-2025-08-07             File Contents            0.0000           0.0000              0.0000           0.0000          0.0000
                               

In [10]:
# Pairwise Cliff's delta across contexts, per (view, tool, model). Mirrors original cell [18].
def pairwise_delta(ai, view_label):
    out = []
    models = sorted(ai['model'].unique())
    for tool in TOOLS:
        prob_col = f'{tool}_ai_probability'
        score_col = prob_col if prob_col in ai.columns else f'{tool}_is_ai'
        for model in models:
            for va, vb in itertools.combinations(VARIATION_ORDER, 2):
                da = ai[(ai['model'] == model) & (ai['variation'] == va)][score_col].dropna().tolist()
                db = ai[(ai['model'] == model) & (ai['variation'] == vb)][score_col].dropna().tolist()
                if not da or not db:
                    continue
                d = cliffs_delta(da, db)
                out.append({
                    'view': view_label,
                    'tool': tool,
                    'model': model,
                    'context_a': VARIATION_MAP[va],
                    'context_b': VARIATION_MAP[vb],
                    'n_a': len(da),
                    'n_b': len(db),
                    'cliffs_delta': round(d, 4),
                    'magnitude': interpret_cliffs_delta(d),
                })
    return pd.DataFrame(out)

delta_frames = []
for key, (ai, hum, label, _) in views.items():
    delta_frames.append(pairwise_delta(ai, key))
delta_all = pd.concat(delta_frames, ignore_index=True)
delta_all.to_csv(OUTPUT_DIR / 'rq1_pairwise_delta_by_view.csv', index=False)
print('rows:', len(delta_all))
display(delta_all.head(10))

rows: 675


,view,tool,model,context_a,context_b,n_a,n_b,cliffs_delta,magnitude
0,original,roberta,Qwen/Qwen2.5-72B-Instruct,Repo Path,PR Title,294,212,-0.5832,Large
1,original,roberta,Qwen/Qwen2.5-72B-Instruct,Repo Path,File Diff,294,305,0.0455,Negligible
2,original,roberta,Qwen/Qwen2.5-72B-Instruct,Repo Path,File Contents,294,302,-0.1987,Small
3,original,roberta,Qwen/Qwen2.5-72B-Instruct,Repo Path,Linked Issue,294,292,-0.0597,Negligible
4,original,roberta,Qwen/Qwen2.5-72B-Instruct,Repo Path,Without File Contents,294,307,0.0098,Negligible
5,original,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title,File Diff,212,305,0.6381,Large
6,original,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title,File Contents,212,302,0.4420,Medium
7,original,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title,Linked Issue,212,292,0.5945,Large
8,original,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title,Without File Contents,212,307,0.6485,Large
9,original,roberta,Qwen/Qwen2.5-72B-Instruct,File Diff,File Contents,305,302,-0.2558,Small


## Did the headline effects from paper Table 4 hold in the subsets?

Table 4 (`tab:rq2_significant_effects`) lists the medium- and large-magnitude context comparisons under the *original* view. For each of those rows we look up the same (tool, model, context-pair) in every view and report side-by-side δ + magnitude. If subset δs stay in the same direction with comparable magnitude, the original finding is robust to detector-failure exclusion.

In [11]:
# Pull the medium+large rows from the original view as our "headline rows to verify"
MAG_RANK = {'Negligible': 0, 'Small': 1, 'Medium': 2, 'Large': 3}
original_delta = delta_all[delta_all['view'] == 'original'].copy()
original_delta['mag_rank'] = original_delta['magnitude'].map(MAG_RANK)
headline = original_delta[original_delta['mag_rank'] >= MAG_RANK['Medium']].drop(columns='mag_rank')
headline = headline.sort_values(['tool', 'model', 'cliffs_delta'], ascending=[True, True, False])
headline.to_csv(OUTPUT_DIR / 'rq1_original_medium_large_rows.csv', index=False)
print(f'Headline rows from original view: {len(headline)}')
display(headline)

Headline rows from original view: 45


,view,tool,model,context_a,context_b,n_a,n_b,cliffs_delta,magnitude
8,original,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title,Without File Contents,212,307,0.6485,Large
5,original,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title,File Diff,212,305,0.6381,Large
7,original,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title,Linked Issue,212,292,0.5945,Large
6,original,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title,File Contents,212,302,0.4420,Medium
0,original,roberta,Qwen/Qwen2.5-72B-Instruct,Repo Path,PR Title,294,212,-0.5832,Large
16,original,roberta,gpt-5-mini-2025-08-07,Repo Path,File Diff,308,306,-0.4600,Medium
37,original,roberta,meta-llama/llama-3.1-70b-instruct,PR Title,Linked Issue,26,152,0.7126,Large
35,original,roberta,meta-llama/llama-3.1-70b-instruct,PR Title,File Diff,26,127,0.6669,Large
42,original,roberta,meta-llama/llama-3.1-70b-instruct,File Contents,Linked Issue,180,152,0.6165,Large
38,original,roberta,meta-llama/llama-3.1-70b-instruct,PR Title,Without File Contents,26,232,0.6097,Large


In [12]:
# Side-by-side: for each headline row, show delta in every view
key_cols = ['tool', 'model', 'context_a', 'context_b']
delta_pivot = delta_all.pivot_table(index=key_cols, columns='view', values='cliffs_delta')
mag_pivot = delta_all.pivot_table(index=key_cols, columns='view', values='magnitude', aggfunc='first')
delta_pivot = delta_pivot[ordered_views]
mag_pivot = mag_pivot[ordered_views]

headline_keys = headline[key_cols].apply(tuple, axis=1).tolist()
delta_compare = delta_pivot.loc[delta_pivot.index.isin(headline_keys)].copy()
mag_compare = mag_pivot.loc[mag_pivot.index.isin(headline_keys)].copy()

# Combined view: "delta (magnitude)"
combined = pd.DataFrame(index=delta_compare.index)
for v in ordered_views:
    combined[v] = [
        f"{d:+.3f} ({m[:3]})" if pd.notna(d) and pd.notna(m) else '—'
        for d, m in zip(delta_compare[v], mag_compare[v])
    ]
combined.to_csv(OUTPUT_DIR / 'rq1_headline_rows_across_views.csv')
display(combined)

original roberta_success roberta-qa_success zerogpt_success common_success
tool       model                             context_a     context_b                                                                                            
roberta    Qwen/Qwen2.5-72B-Instruct         PR Title      File Contents          +0.442 (Med)    +0.432 (Med)       +0.432 (Med)    +0.442 (Med)   +0.432 (Med)
                                                           File Diff              +0.638 (Lar)    +0.627 (Lar)       +0.627 (Lar)    +0.638 (Lar)   +0.627 (Lar)
                                                           Linked Issue           +0.595 (Lar)    +0.616 (Lar)       +0.616 (Lar)    +0.595 (Lar)   +0.616 (Lar)
                                                           Without File Contents  +0.648 (Lar)    +0.649 (Lar)       +0.649 (Lar)    +0.648 (Lar)   +0.649 (Lar)
                                             Repo Path     PR Title               -0.583 (Lar)    -0.593 (Lar)       -0.593 (Lar)    -0.583 (Lar)   -0.593 (Lar)
           gpt-5-mini-2025-08-07             Repo Path     File Diff              -0.460 (Med)    -0.438 (Med)       -0.438 (Med)    -0.460 (Med)   -0.438 (Med)
           meta-llama/llama-3.1-70b-instruct File Contents Linked Issue           +0.617 (Lar)    +0.624 (Lar)       +0.624 (Lar)    +0.617 (Lar)   +0.624 (Lar)
                                                           Without File Contents  +0.546 (Lar)    +0.550 (Lar)       +0.550 (Lar)    +0.546 (Lar)   +0.550 (Lar)
                                             File Diff     File Contents          -0.629 (Lar)    -0.626 (Lar)       -0.626 (Lar)    -0.629 (Lar)   -0.626 (Lar)
                                             PR Title      File Diff              +0.667 (Lar)    +0.776 (Lar)       +0.776 (Lar)    +0.667 (Lar)   +0.776 (Lar)
                                                           Linked Issue           +0.713 (Lar)    +0.846 (Lar)       +0.846 (Lar)    +0.713 (Lar)   +0.846 (Lar)
                                                           Without File Contents  +0.610 (Lar)    +0.741 (Lar)       +0.741 (Lar)    +0.610 (Lar)   +0.741 (Lar)
                                             Repo Path     File Diff              +0.406 (Med)    +0.401 (Med)       +0.401 (Med)    +0.406 (Med)   +0.401 (Med)
                                                           Linked Issue           +0.373 (Med)    +0.381 (Med)       +0.381 (Med)    +0.373 (Med)   +0.381 (Med)
roberta-qa Qwen/Qwen2.5-72B-Instruct         File Contents Linked Issue           +0.758 (Lar)    +0.772 (Lar)       +0.772 (Lar)    +0.758 (Lar)   +0.772 (Lar)
                                                           Without File Contents  +0.567 (Lar)    +0.565 (Lar)       +0.565 (Lar)    +0.567 (Lar)   +0.565 (Lar)
                                             File Diff     File Contents          -0.455 (Med)    -0.446 (Med)       -0.446 (Med)    -0.455 (Med)   -0.446 (Med)
                                                           Linked Issue           +0.402 (Med)    +0.438 (Med)       +0.438 (Med)    +0.402 (Med)   +0.438 (Med)
                                             PR Title      Linked Issue           +0.696 (Lar)    +0.711 (Lar)       +0.711 (Lar)    +0.696 (Lar)   +0.711 (Lar)
                                                           Without File Contents  +0.445 (Med)    +0.436 (Med)       +0.436 (Med)    +0.445 (Med)   +0.436 (Med)
                                             Repo Path     File Contents          -0.586 (Lar)    -0.620 (Lar)       -0.620 (Lar)    -0.586 (Lar)   -0.620 (Lar)
                                                           PR Title               -0.465 (Med)    -0.501 (Lar)       -0.501 (Lar)    -0.465 (Med)   -0.501 (Lar)
           meta-llama/llama-3.1-70b-instruct File Contents Linked Issue           +0.706 (Lar)    +0.727 (Lar)       +0.727 (Lar)    +0.706 (Lar)   +0.727 (Lar)
                                                           Wi

In [13]:
# Flag any headline row whose magnitude *dropped* by 2+ levels in the per-tool successful view.
# (Per-tool view = the view that matches the row's tool. For zerogpt, that's zerogpt_success;
# for the two roberta variants, it's the matching {tool}_success view.)
def per_tool_view(tool):
    return f'{tool}_success'

robustness_rows = []
for _, row in headline.iterrows():
    tool, model, ca, cb = row['tool'], row['model'], row['context_a'], row['context_b']
    orig_d, orig_m = row['cliffs_delta'], row['magnitude']
    pt_view = per_tool_view(tool)
    matching = delta_all[
        (delta_all['view'] == pt_view)
        & (delta_all['tool'] == tool)
        & (delta_all['model'] == model)
        & (delta_all['context_a'] == ca)
        & (delta_all['context_b'] == cb)
    ]
    common = delta_all[
        (delta_all['view'] == 'common_success')
        & (delta_all['tool'] == tool)
        & (delta_all['model'] == model)
        & (delta_all['context_a'] == ca)
        & (delta_all['context_b'] == cb)
    ]
    pt_d = matching['cliffs_delta'].iloc[0] if not matching.empty else np.nan
    pt_m = matching['magnitude'].iloc[0] if not matching.empty else None
    cs_d = common['cliffs_delta'].iloc[0] if not common.empty else np.nan
    cs_m = common['magnitude'].iloc[0] if not common.empty else None
    sign_flip_pt = (not np.isnan(pt_d)) and (np.sign(pt_d) != np.sign(orig_d))
    drop_pt = (pt_m is not None) and (MAG_RANK[orig_m] - MAG_RANK[pt_m] >= 2)
    sign_flip_cs = (not np.isnan(cs_d)) and (np.sign(cs_d) != np.sign(orig_d))
    drop_cs = (cs_m is not None) and (MAG_RANK[orig_m] - MAG_RANK[cs_m] >= 2)
    robustness_rows.append({
        'tool': tool,
        'model': model,
        'comparison': f'{ca} vs {cb}',
        'orig_delta': round(orig_d, 3),
        'orig_mag': orig_m,
        'pertool_delta': round(pt_d, 3) if not np.isnan(pt_d) else None,
        'pertool_mag': pt_m,
        'pertool_sign_flip': sign_flip_pt,
        'pertool_mag_drop_2plus': drop_pt,
        'common_delta': round(cs_d, 3) if not np.isnan(cs_d) else None,
        'common_mag': cs_m,
        'common_sign_flip': sign_flip_cs,
        'common_mag_drop_2plus': drop_cs,
    })

robust = pd.DataFrame(robustness_rows)
robust.to_csv(OUTPUT_DIR / 'rq1_robustness_check.csv', index=False)
print('Total headline rows checked:', len(robust))
print('Per-tool sign flips:', int(robust['pertool_sign_flip'].sum()))
print('Per-tool magnitude drops of 2+:', int(robust['pertool_mag_drop_2plus'].sum()))
print('Common-subset sign flips:', int(robust['common_sign_flip'].sum()))
print('Common-subset magnitude drops of 2+:', int(robust['common_mag_drop_2plus'].sum()))
display(robust)

Total headline rows checked: 45
Per-tool sign flips: 0
Per-tool magnitude drops of 2+: 0
Common-subset sign flips: 0
Common-subset magnitude drops of 2+: 1


,tool,model,comparison,orig_delta,orig_mag,pertool_delta,pertool_mag,pertool_sign_flip,pertool_mag_drop_2plus,common_delta,common_mag,common_sign_flip,common_mag_drop_2plus
0,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title vs Without File Contents,0.648,Large,0.649,Large,False,False,0.649,Large,False,False
1,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title vs File Diff,0.638,Large,0.628,Large,False,False,0.628,Large,False,False
2,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title vs Linked Issue,0.595,Large,0.616,Large,False,False,0.616,Large,False,False
3,roberta,Qwen/Qwen2.5-72B-Instruct,PR Title vs File Contents,0.442,Medium,0.432,Medium,False,False,0.432,Medium,False,False
4,roberta,Qwen/Qwen2.5-72B-Instruct,Repo Path vs PR Title,-0.583,Large,-0.593,Large,False,False,-0.593,Large,False,False
5,roberta,gpt-5-mini-2025-08-07,Repo Path vs File Diff,-0.460,Medium,-0.438,Medium,False,False,-0.438,Medium,False,False
6,roberta,meta-llama/llama-3.1-70b-instruct,PR Title vs Linked Issue,0.713,Large,0.846,Large,False,False,0.846,Large,False,False
7,roberta,meta-llama/llama-3.1-70b-instruct,PR Title vs File Diff,0.667,Large,0.776,Large,False,False,0.776,Large,False,False
8,roberta,meta-llama/llama-3.1-70b-instruct,File Contents vs Linked Issue,0.617,Large,0.624,Large,False,False,0.624,Large,False,False
9,roberta,meta-llama/llama-3.1-70b-instruct,PR Title vs Without File Contents,0.610,Large,0.741,Large,False,False,0.741,Large,False,False


## Decision summary

Read this cell to decide whether to add a sensitivity paragraph to Threats-to-Validity and whether RQ1 wording needs revision.

- If `pertool_sign_flip == 0` and `pertool_mag_drop_2plus == 0`: the original RQ1 effects survive when failed detections are excluded — story is robust, add a one-paragraph sensitivity note.
- If `common_sign_flip` or `common_mag_drop_2plus` is non-zero: the strictest view weakens some effects — list those rows in the paper and qualify the relevant claims.
- Compare `prelim_rq_f1_pivot.csv` — if F1 changes materially for any (tool, model) on P-3 between `original` and `T_success`, mention it in the Prelim-RQ discussion.

In [14]:
# Per-tool F1 deltas on P-3 (original vs T_success)
prelim_f1 = prelim_all.pivot_table(index=['tool', 'model'], columns='view', values='F1-Score')
deltas = pd.DataFrame(index=prelim_f1.index)
for tool in TOOLS:
    pt_col = f'{tool}_success'
    if pt_col in prelim_f1.columns:
        mask = prelim_f1.index.get_level_values('tool') == tool
        deltas.loc[mask, 'orig_F1'] = prelim_f1.loc[mask, 'original']
        deltas.loc[mask, 'subset_F1'] = prelim_f1.loc[mask, pt_col]
        deltas.loc[mask, 'F1_diff'] = (prelim_f1.loc[mask, pt_col] - prelim_f1.loc[mask, 'original']).round(4)
deltas = deltas.dropna(how='all')
deltas.to_csv(OUTPUT_DIR / 'prelim_rq_f1_pertool_diff.csv')
display(deltas)

orig_F1  subset_F1  F1_diff
tool       model                                                         
roberta    Qwen/Qwen2.5-72B-Instruct           0.3351     0.3492   0.0141
           gpt-5-mini-2025-08-07               0.0129     0.0151   0.0022
           meta-llama/llama-3.1-70b-instruct   0.1612     0.3750   0.2138
roberta-qa Qwen/Qwen2.5-72B-Instruct           0.5904     0.6096   0.0192
           gpt-5-mini-2025-08-07               0.0000     0.0000   0.0000
           meta-llama/llama-3.1-70b-instruct   0.1502     0.3286   0.1784
zerogpt    Qwen/Qwen2.5-72B-Instruct           0.6095     0.6095   0.0000
           gpt-5-mini-2025-08-07               0.1829     0.1829   0.0000
           meta-llama/llama-3.1-70b-instruct   0.0256     0.0256   0.0000